# 3.4 词性标注

## 学习目标

- 理解词性标注的概念
- 掌握jieba词性标注的使用
- 学会根据词性筛选词语
- 了解词性标注的应用场景

## 3.4.1 什么是词性标注？

**词性标注**（Part-of-Speech Tagging）是给每个词标注其语法属性的过程。

### 常见词性

- **n**：名词（noun）- 手机、电脑、学生
- **v**：动词（verb）- 学习、工作、吃饭
- **a**：形容词（adjective）- 好的、漂亮、聪明
- **d**：副词（adverb）- 很、非常、特别
- **m**：数词（numeral）- 一、二、三
- **q**：量词（quantifier）- 个、台、本

### 为什么需要词性标注？

- 理解词语的语法功能
- 提取特定类型的词（如只要名词）
- 提高文本分析的准确性

## 3.4.2 为什么需要pseg？

### 对比：普通分词 vs 词性标注

In [ ]:
import jieba

text = "我喜欢学习自然语言处理技术"

print(f"原文：{text}\n")
print("="*50)

# 方法1：普通分词（之前学过的）
print("【方法1】普通分词 - 只能切分词语：")
words = jieba.lcut(text)
print(" / ".join(words))

**问题**：我们不知道每个词是什么类型（名词？动词？）

### 解决方案：使用pseg模块

**pseg = Part-of-Speech Segmentation（词性分词）**

- `jieba` 模块：只做分词
- `jieba.posseg` 模块（简称pseg）：分词 + 标注词性

### pseg.cut() 函数说明

**函数作用**：对文本进行分词，并为每个词标注词性

**基本用法**：
```python
words = pseg.cut(text)
```

**返回结果**：返回一个生成器，每个元素是一个 (词语, 词性) 的配对

**使用方式**：
- 可以用 `for` 循环遍历：`for word, flag in words:`
- `word` 是分词结果（字符串）
- `flag` 是词性标记（字符串，如 'n'、'v'、'a'）

In [ ]:
import jieba.posseg as pseg

# 方法2：词性标注 - 不仅切分，还告诉你每个词的类型
print("\n【方法2】词性标注 - 知道每个词的类型：")
print("="*50)

words = pseg.cut(text)

for word, flag in words:
    print(f"{word:10s} → {flag:5s}")

**结果分析**：现在我们知道'喜欢'是动词(v)，'技术'是名词(n)！

### 理解词性代码

让我们把词性代码"翻译"成中文：

In [ ]:
# 词性代码对照表
pos_explain = {
    'r': '代词（我、你、他）',
    'v': '动词（喜欢、学习、处理）',
    'l': '习语（自然语言）',
    'n': '名词（技术、手机、学生）'
}

print("词性代码含义：\n")
words = pseg.cut(text)

for word, flag in words:
    meaning = pos_explain.get(flag, '其他词性')
    print(f"'{word}' → {flag} → {meaning}")

### 为什么要知道词性？

**实际应用举例**：假设你要分析商品评论，只想提取"商品名称"（名词）

In [ ]:
review = "这个手机拍照很清晰，电池续航能力强"

print(f"评论：{review}\n")

# 使用pseg提取所有名词
words = pseg.cut(review)
nouns = [word for word, flag in words if flag.startswith('n')]

print("提取的商品相关名词：")
print(" / ".join(nouns))

**分析**：有了词性标注，我们可以精准提取需要的词！

### 常用词性对照表

In [ ]:
# 词性对照表
pos_dict = {
    'n': '名词',
    'v': '动词', 
    'a': '形容词',
    'd': '副词',
    'r': '代词',
    'm': '数词',
    'q': '量词',
    'p': '介词',
    'c': '连词'
}

print("常用词性对照表：\n")
for pos, name in pos_dict.items():
    print(f"{pos:5s} - {name}")

### 小结

**记住这三点：**

1. **jieba.lcut()** - 只分词，不知道词性
2. **jieba.posseg.cut()** - 分词 + 标注词性
3. **pseg是posseg的简写** - 为了方便使用，我们写成 `import jieba.posseg as pseg`

**什么时候用pseg？**
- 需要提取特定类型的词（如只要名词）
- 需要区分词语的语法功能
- 做更精细的文本分析

## 3.4.3 根据词性筛选词语

### 提取名词

In [ ]:
text = "这款智能手机的屏幕很大，电池续航能力强"

print(f"原文：{text}\n")

# 提取所有名词
words = pseg.cut(text)
nouns = [word for word, flag in words if flag.startswith('n')]

print("提取的名词：")
print(" / ".join(nouns))

**分析**：名词通常是关键信息的载体

### 提取动词

In [ ]:
text = "用户可以下载、安装和使用这个软件"

print(f"原文：{text}\n")

# 提取所有动词
words = pseg.cut(text)
verbs = [word for word, flag in words if flag.startswith('v')]

print("提取的动词：")
print(" / ".join(verbs))

**分析**：动词表示动作和行为

### 提取形容词

In [ ]:
text = "这个产品质量好，价格便宜，性能强大"

print(f"原文：{text}\n")

# 提取所有形容词
words = pseg.cut(text)
adjs = [word for word, flag in words if flag.startswith('a')]

print("提取的形容词：")
print(" / ".join(adjs))

**分析**：形容词常用于情感分析

## 3.4.4 构建词性过滤函数

In [ ]:
def extract_by_pos(text, pos_list=['n', 'v', 'a']):
    """
    根据词性提取词语
    
    参数：
        text: 原始文本
        pos_list: 要提取的词性列表
    返回：
        符合词性的词列表
    """
    words = pseg.cut(text)
    result = []
    
    for word, flag in words:
        for pos in pos_list:
            if flag.startswith(pos):
                result.append(word)
                break
    
    return result

### 测试函数

In [ ]:
text = "我非常喜欢这个智能手机，它的拍照功能很强大"

print(f"原文：{text}\n")

# 只提取名词
nouns = extract_by_pos(text, ['n'])
print(f"名词：{' / '.join(nouns)}")

# 提取名词和形容词
result = extract_by_pos(text, ['n', 'a'])
print(f"名词+形容词：{' / '.join(result)}")

# 提取名词、动词和形容词
result = extract_by_pos(text, ['n', 'v', 'a'])
print(f"名词+动词+形容词：{' / '.join(result)}")

## 3.4.5 应用场景

### 场景1：关键词提取

通常只提取名词和形容词作为关键词。

In [ ]:
text = "这款新推出的智能手表功能强大，设计精美，价格合理"

print(f"原文：{text}\n")

# 提取关键词（名词+形容词）
keywords = extract_by_pos(text, ['n', 'a'])

print("关键词：")
print(' / '.join(keywords))

**分析**：这些词最能代表文本的主要内容

### 场景2：情感分析

形容词和副词常表达情感倾向。

In [ ]:
text = "这部电影非常精彩，演员表演特别出色，强烈推荐"

print(f"原文：{text}\n")

# 提取情感词（形容词+副词）
sentiment_words = extract_by_pos(text, ['a', 'd'])

print("情感词：")
print(' / '.join(sentiment_words))

**分析**：这些词表达了正面情感

## 3.4.6 课程总结

### 本节学习了：

1. **词性标注的概念**
   - 标注词语的语法属性
   - 常见词性：名词、动词、形容词等

2. **使用jieba词性标注**
   - `jieba.posseg` 模块
   - `pseg.cut()` 方法

3. **根据词性筛选**
   - 提取特定词性的词语
   - 构建通用过滤函数

4. **应用场景**
   - 关键词提取：名词+形容词
   - 情感分析：形容词+副词
   - 实体识别：名词

### 注意事项

- ⚠️ 词性标注不是100%准确
- ⚠️ 根据任务选择合适的词性
- ⚠️ 可以组合使用多个词性

### 练习建议

- 尝试提取不同词性的词语
- 分析不同词性在文本中的作用
- 将词性标注与其他预处理步骤结合